In [1]:
import os
import zstandard  # pip install zstandard
from tqdm import tqdm
import random
import json

files_processed_to_text = True




In [25]:
from datetime import datetime
def showTime():
    return str("["+datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')+" UTC]")

In [2]:
def zst_files_in_dir(directory):
    """List all .zst files in a directory."""
    files = []
    for filename in os.listdir(directory):
        if filename.endswith(".zst") and os.path.isfile(os.path.join(directory, filename)):
            files.append(filename)
    return files

In [3]:
def decompress_zst_to_text(input_file, vocab):
    """
    Decompresses a .zst file containing JSONL (JSON lines) format, extracts the "text" value from each line.
    
    Parameters:
        input_file_path (str): Path to the .zst file.
        vocab (set, optional): The vocabulary set to update with characters.
    
    Yields:
        str: The extracted text from each JSON line.
    """
    with open(input_file, "rb") as infile:
        dctx = zstandard.ZstdDecompressor()
        with dctx.stream_reader(infile) as reader:
            current_line = ""
            while True:
                chunk = reader.read(16384).decode("utf-8", errors="replace")  # Read in 16KB chunks
                if not chunk:
                    break
                current_line += chunk
                # Split into lines (handles partial lines)
                lines = current_line.split("\n")
                current_line = lines.pop() if lines else ""  # Save partial line for next iteration
                # Process each line
                for line in lines:
                    if not line.strip():
                        continue
                    try:
                        data = json.loads(line)
                        text = data.get("text", "")
                        if vocab is not None:
                            vocab.update(set(text))
                        yield text.strip()
                    except json.JSONDecodeError as e:
                        print(f"JSON Error: {e} on line: {line[:50]}...")


In [4]:
folder_path = "openwebtext2"
output_file_train = "output_train_v3.txt"
output_file_val = "output_val_v3.txt"
vocab_file = "vocab_v3.txt"


In [5]:
# Gather files
files = zst_files_in_dir(folder_path)
total_files = len(files)
print(f"Total files: {total_files}")
print(files)

Total files: 179
['2005-06.jsonl.zst', '2005-07.jsonl.zst', '2005-08.jsonl.zst', '2005-09.jsonl.zst', '2005-10.jsonl.zst', '2005-11.jsonl.zst', '2005-12.jsonl.zst', '2006-01.jsonl.zst', '2006-02.jsonl.zst', '2006-03.jsonl.zst', '2006-04.jsonl.zst', '2006-05.jsonl.zst', '2006-06.jsonl.zst', '2006-07.jsonl.zst', '2006-08.jsonl.zst', '2006-09.jsonl.zst', '2006-10.jsonl.zst', '2006-11.jsonl.zst', '2006-12.jsonl.zst', '2007-01.jsonl.zst', '2007-02.jsonl.zst', '2007-03.jsonl.zst', '2007-04.jsonl.zst', '2007-05.jsonl.zst', '2007-06.jsonl.zst', '2007-07.jsonl.zst', '2007-08.jsonl.zst', '2007-09.jsonl.zst', '2007-10.jsonl.zst', '2007-11.jsonl.zst', '2007-12.jsonl.zst', '2008-01.jsonl.zst', '2008-02.jsonl.zst', '2008-03.jsonl.zst', '2008-04.jsonl.zst', '2008-05.jsonl.zst', '2008-06.jsonl.zst', '2008-07.jsonl.zst', '2008-08.jsonl.zst', '2008-09.jsonl.zst', '2008-10.jsonl.zst', '2008-11.jsonl.zst', '2008-12.jsonl.zst', '2009-01.jsonl.zst', '2009-02.jsonl.zst', '2009-03.jsonl.zst', '2009-04.jsonl.z

In [6]:
# Shuffle files randomly 
random.seed(42)  # Optional: Set seed for reproducibility
random.shuffle(files)  # Shuffle in-place
print(files)

['2018-02.jsonl.zst', '2009-11.jsonl.zst', '2006-09.jsonl.zst', '2008-12.jsonl.zst', '2006-07.jsonl.zst', '2008-06.jsonl.zst', '2010-06.jsonl.zst', '2015-08.jsonl.zst', '2010-07.jsonl.zst', '2007-01.jsonl.zst', '2010-11.jsonl.zst', '2007-12.jsonl.zst', '2018-05.jsonl.zst', '2005-08.jsonl.zst', '2016-08.jsonl.zst', '2016-09.jsonl.zst', '2015-06.jsonl.zst', '2018-09.jsonl.zst', '2019-07.jsonl.zst', '2010-12.jsonl.zst', '2013-04.jsonl.zst', '2019-11.jsonl.zst', '2007-07.jsonl.zst', '2016-06.jsonl.zst', '2017-10.jsonl.zst', '2018-08.jsonl.zst', '2019-12.jsonl.zst', '2011-08.jsonl.zst', '2017-03.jsonl.zst', '2017-07.jsonl.zst', '2006-08.jsonl.zst', '2009-10.jsonl.zst', '2016-02.jsonl.zst', '2014-02.jsonl.zst', '2009-02.jsonl.zst', '2015-09.jsonl.zst', '2011-03.jsonl.zst', '2012-02.jsonl.zst', '2018-10.jsonl.zst', '2005-06.jsonl.zst', '2015-01.jsonl.zst', '2006-10.jsonl.zst', '2016-10.jsonl.zst', '2015-11.jsonl.zst', '2013-10.jsonl.zst', '2010-10.jsonl.zst', '2018-06.jsonl.zst', '2012-05.jso

In [7]:
# Split files into train/val (90%/10%)
split_index = int(total_files * 0.9)
files_train = files[:split_index]
files_val = files[split_index:]
vocab = set()

In [8]:
# Process training files
if files_processed_to_text == False:
    with open(output_file_train, "w", encoding="utf-8") as outf:
        for filename in tqdm(files_train, total=len(files_train), desc="Processing Train"):
            file_path = os.path.join(folder_path, filename)
            try:
                for text_line in decompress_zst_to_text(file_path, vocab):
                    outf.write(text_line.strip() + "\n")  # Write only the text line
            except Exception as e:
                print(f"Error processing {file_path}: {e}")

In [9]:
# Process validation files
if files_processed_to_text == False:
    with open(output_file_val, "w", encoding="utf-8") as outf:
        for filename in tqdm(files_val, total=len(files_val), desc="Processing Val"):
            file_path = os.path.join(folder_path, filename)
            try:
                for text_line in decompress_zst_to_text(file_path, vocab):
                    outf.write(text_line.strip() + "\n")  # Write only the text line
            except Exception as e:
                print(f"Error processing {file_path}: {e}")


In [10]:
#load sequence
with open(output_file_val, "r", encoding="utf-8") as f:
    number_of_characters_to_read = 1_000_000
    text_sequence = f.read(number_of_characters_to_read)

len(text_sequence)

1000000

In [11]:
# Karpathy minBPE repository
from minbpe import RegexTokenizer

tokenizer = RegexTokenizer()
tokenizer.train(text_sequence, vocab_size=1024)

In [12]:
vocab = tokenizer.vocab
vocab

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'

In [13]:
tokenizer.encode("Hello, world! I like apple juice - I drink it every day. Isn't that too much?")

[72,
 464,
 111,
 44,
 916,
 33,
 333,
 693,
 601,
 297,
 459,
 117,
 477,
 1007,
 333,
 743,
 857,
 343,
 833,
 289,
 331,
 46,
 333,
 115,
 110,
 785,
 318,
 286,
 111,
 961,
 63]

In [14]:
tokenizer.decode([72,
 464,
 111,
 44,
 916,
 33,
 333,
 693,
 601,
 297,
 459,
 117,
 477,
 1007,
 333,
 743,
 857,
 343,
 833,
 289,
 331,
 46,
 333,
 115,
 110,
 785,
 318,
 286,
 111,
 961,
 63])

"Hello, world! I like apple juice - I drink it every day. Isn't that too much?"

In [15]:
max_vocab_id = list(tokenizer.vocab.keys())[-1]
tokenizer.special_tokens = {
    "<|startoftext|>": max_vocab_id + 1,
    "<|separator|>": max_vocab_id + 2,
    "<|endoftext|>": max_vocab_id + 3,
    "<|unk|>": max_vocab_id + 4,
    "<|padding|>": max_vocab_id + 5
}

In [22]:
tokenizer_output_dir = "output_v6/tokenizer"
import os
if not os.path.exists(tokenizer_output_dir):
    os.makedirs(tokenizer_output_dir)

tokenizer_path = os.path.join(tokenizer_output_dir, "darija_tokenizer")
tokenizer.save(file_prefix=tokenizer_path)

In [ ]:
# Encoding the sequence of text

from minbpe import RegexTokenizer

tokenizer = RegexTokenizer()
tokenizer.load(model_file=tokenizer_path+".model")

In [27]:
# Encode the data in batches

encoded_text_sequence = []
batch_size = 100_000_000
file_path = "output_val_v3.txt"

with open(file_path, "r", encoding="utf-8") as f:
    while True:
        chunk = f.read(batch_size)
        if not chunk:
            break
        batch_tokens = tokenizer.encode(chunk)
        encoded_text_sequence.extend(batch_tokens)
        print(f"{showTime()} Processed {len(encoded_text_sequence)} tokens so far.")

print(f"Total tokens: {len(encoded_text_sequence)}")

[2025-04-27 22:00:06.567947 UTC] Processed 43972653 tokens so far.
[2025-04-27 22:03:51.935770 UTC] Processed 87930687 tokens so far.
[2025-04-27 22:07:35.993889 UTC] Processed 131101901 tokens so far.
[2025-04-27 22:11:19.872137 UTC] Processed 174941227 tokens so far.
[2025-04-27 22:15:03.596542 UTC] Processed 218990051 tokens so far.
[2025-04-27 22:18:49.688220 UTC] Processed 263053839 tokens so far.
[2025-04-27 22:22:33.683296 UTC] Processed 306664429 tokens so far.
[2025-04-27 22:26:18.820127 UTC] Processed 348925909 tokens so far.
[2025-04-27 22:30:11.805303 UTC] Processed 390673018 tokens so far.
[2025-04-27 22:34:04.602860 UTC] Processed 433258045 tokens so far.
[2025-04-27 22:37:50.889300 UTC] Processed 476315828 tokens so far.
[2025-04-27 22:41:40.110096 UTC] Processed 519267403 tokens so far.
[2025-04-27 22:45:31.861220 UTC] Processed 562433171 tokens so far.
[2025-04-27 22:49:24.397713 UTC] Processed 606137308 tokens so far.
[2025-04-27 22:53:13.125417 UTC] Processed 6499788

In [29]:
import numpy as np


encoder_output_dir = "output_v6/encoded_data"
import os
if not os.path.exists(encoder_output_dir):
    os.makedirs(encoder_output_dir)

output_path = os.path.join(encoder_output_dir, "encoded_output_val_v3.npy")
np.save(output_path, np.array(encoded_text_sequence, dtype=np.int64))

# Free up memory
del encoded_text_sequence